# Bifrost API test

Smoke tests for the Bifrost gateway (OpenAI-compatible) in front of vLLM: model list, chat, streaming, Qwen3 thinking on/off,
JSON output, tool calling, embeddings, rerank, and a small concurrency test.

**Setup:** `BIFROST_BASE_URL` and `BIFROST_API_KEY` live in `../.env` (never in this notebook).
The gateway is on the internal network: connect to the VPN / office network first.
Each test records its outcome in `results`; the last cell prints a summary.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
BASE_URL = os.environ["BIFROST_BASE_URL"]
API_KEY = os.environ["BIFROST_API_KEY"]
# Bifrost virtual keys go in x-bf-vk (Bearer is also accepted); send both
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=120, default_headers={"x-bf-vk": API_KEY})
results = {}


def record(name, ok, detail=""):
    results[name] = ("OK" if ok else "FAIL") + (f" - {detail}" if detail else "")
    print(name, "->", results[name])

## 1. Reachability and model list

In [ ]:
t0 = time.time()
models = [m.id for m in client.models.list().data]
record("list_models", bool(models), f"{len(models)} models in {time.time() - t0:.2f}s")
for m in models:
    print(" ", m)

Pick the models by role. Bifrost ids may carry a provider prefix (e.g. `vllm/qwen3.6-27b`), so match on the name. Override by hand if needed.

In [ ]:
def pick(fragment, exclude=()):
    found = [m for m in models if fragment in m.lower() and not any(x in m.lower() for x in exclude)]
    assert found, f"no model containing {fragment!r}: {models}"
    return found[0]


CHAT_MODEL = pick("qwen3", exclude=("embed", "rerank"))
EMBED_MODELS = [m for m in models if "embed" in m.lower()]
RERANK_MODEL = pick("rerank")
print("chat:", CHAT_MODEL, "| embed:", EMBED_MODELS, "| rerank:", RERANK_MODEL)

## 2. Chat completion

In [ ]:
t0 = time.time()
r = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "En une phrase : à partir de quoi fabrique-t-on le DAP ?"}],
    max_tokens=512,
)
latency = time.time() - t0
print(r.choices[0].message.content)
print(r.usage)
record("chat", bool(r.choices[0].message.content), f"{latency:.1f}s, {r.usage.completion_tokens} output tokens")

## 3. Streaming (time to first token, throughput)

In [ ]:
t0, first, chunks = time.time(), None, []
stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "List 5 phosphate fertilizers with one line each."}],
    max_tokens=400,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        first = first or time.time()
        chunks.append(delta)
        print(delta, end="", flush=True)
total = time.time() - t0
text = "".join(chunks)
print(f"\n\nTTFT {first - t0:.2f}s, total {total:.1f}s, ~{len(text) / 4 / (total - (first - t0)):.0f} tokens/s (chars/4 estimate)")
record("streaming", bool(text), f"TTFT {first - t0:.2f}s")

## 4. Qwen3 thinking mode

vLLM serves Qwen3 with thinking on by default. Reasoning may come back in `reasoning_content` (if vLLM runs with a reasoning parser)
or inline as `<think>…</think>`. `chat_template_kwargs.enable_thinking=False` turns it off; check whether Bifrost forwards it.

In [ ]:
question = [{"role": "user", "content": "What is 17 * 23? Answer with the number only."}]
for thinking in (True, False):
    t0 = time.time()
    r = client.chat.completions.create(
        model=CHAT_MODEL, messages=question, max_tokens=1024,
        extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
    )
    msg = r.choices[0].message
    reasoning = getattr(msg, "reasoning_content", None) or (msg.model_extra or {}).get("reasoning_content")
    print(f"thinking={thinking}: {time.time() - t0:.1f}s, {r.usage.completion_tokens} tokens")
    print("  content:", repr(msg.content[:200]))
    print("  reasoning_content:", repr((reasoning or "")[:200]))
record("thinking_toggle", True, "inspect output above: content should be 391; fewer tokens with thinking off")

## 5. JSON output (schema-constrained)

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "product": {"type": "string"},
        "location": {"type": "string"},
        "incoterm": {"type": "string", "enum": ["fob", "cfr", "cif", "fca", "unknown"]},
        "low": {"type": "number"},
        "high": {"type": "number"},
    },
    "required": ["product", "location", "incoterm", "low", "high"],
    "additionalProperties": False,
}
text = "Mainland China: DAP prices are unchanged at $695-720/t fob. A lack of export availability is keeping the market quiet."
try:
    r = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "system", "content": "Extract the price assessment."}, {"role": "user", "content": text}],
        response_format={"type": "json_schema", "json_schema": {"name": "price", "schema": schema, "strict": True}},
        max_tokens=512,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    data = json.loads(r.choices[0].message.content)
    print(data)
    record("json_schema", data.get("low") == 695 and data.get("high") == 720, json.dumps(data))
except Exception as e:  # a test notebook: report the failure and keep going
    record("json_schema", False, str(e)[:200])

## 6. Tool calling (needs vLLM `--enable-auto-tool-choice` and a tool parser)

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "price_monthly",
        "description": "Monthly average price of a product on a route.",
        "parameters": {
            "type": "object",
            "properties": {"product": {"type": "string"}, "location": {"type": "string"}, "incoterm": {"type": "string"}},
            "required": ["product", "location", "incoterm"],
        },
    },
}]
try:
    r = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": "What was the monthly DAP fob price in Morocco? Use the tool."}],
        tools=tools,
        max_tokens=1024,
    )
    calls = r.choices[0].message.tool_calls or []
    for c in calls:
        print(c.function.name, c.function.arguments)
    record("tool_calling", bool(calls), f"{len(calls)} call(s)" if calls else f"no tool call; content={r.choices[0].message.content[:120]!r}")
except Exception as e:  # a test notebook: report the failure and keep going
    record("tool_calling", False, str(e)[:200])

## 7. Embeddings (dimension, latency, FR/EN similarity sanity check)

In [ ]:
sentences = [
    "Le prix du DAP a fortement augmenté pendant la guerre au Moyen-Orient.",   # FR
    "DAP prices rose sharply during the Middle East war.",                      # EN, same meaning
    "Ammonia is made from natural gas.",                                        # related domain, different meaning
    "The football match ended in a draw.",                                      # unrelated
]


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


for model in EMBED_MODELS:
    try:
        t0 = time.time()
        vectors = [np.array(d.embedding) for d in client.embeddings.create(model=model, input=sentences).data]
        sims = [cosine(vectors[0], v) for v in vectors[1:]]
        print(f"{model}: dim={len(vectors[0])}, {time.time() - t0:.2f}s, sim(FR, EN same)={sims[0]:.3f}, related={sims[1]:.3f}, unrelated={sims[2]:.3f}")
        record(f"embed:{model}", sims[0] > sims[1] > sims[2], f"dim {len(vectors[0])}")
    except Exception as e:  # a test notebook: report the failure and keep going
        record(f"embed:{model}", False, str(e)[:200])

## 8. Rerank

Rerank is not part of the OpenAI API. vLLM exposes `/rerank` and `/v1/rerank` (Jina/Cohere style); whether Bifrost forwards it depends on its
configuration, so this cell tries both paths.

In [ ]:
query = "Why did DAP prices rise in 2026?"
documents = [
    "Hormuz disruptions blocked Saudi phosphate exports and tightened the DAP market.",
    "Morocco's football team won the match.",
    "Higher sulphur costs from Middle East supply disruptions pushed Chinese DAP prices up.",
    "Urea is a nitrogen fertilizer.",
]
root = BASE_URL.removesuffix("/v1")
for path in (f"{BASE_URL}/rerank", f"{root}/rerank"):
    try:
        resp = requests.post(path, headers={"Authorization": f"Bearer {API_KEY}", "x-bf-vk": API_KEY},
                             json={"model": RERANK_MODEL, "query": query, "documents": documents}, timeout=60)
        print(path, resp.status_code)
        if resp.ok:
            ranked = sorted(resp.json()["results"], key=lambda x: -x["relevance_score"])
            for x in ranked:
                print(f"  {x['relevance_score']:.3f}  {documents[x['index']]}")
            record("rerank", ranked[0]["index"] in (0, 2), path)
            break
        print("  ", resp.text[:200])
    except Exception as e:  # a test notebook: report the failure and keep going
        print(path, "error:", str(e)[:200])
results.setdefault("rerank", "FAIL - no rerank path answered")

## 9. Small concurrency test (8 parallel short requests)

In [ ]:
def one(i):
    t0 = time.time()
    client.chat.completions.create(
        model=CHAT_MODEL, messages=[{"role": "user", "content": f"Say the number {i}."}], max_tokens=20,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return time.time() - t0


t0 = time.time()
with ThreadPoolExecutor(8) as pool:
    latencies = list(pool.map(one, range(8)))
wall = time.time() - t0
print(f"wall {wall:.1f}s, per-request min {min(latencies):.1f}s / median {sorted(latencies)[4]:.1f}s / max {max(latencies):.1f}s")
record("concurrency_8", True, f"wall {wall:.1f}s, max {max(latencies):.1f}s")

## Summary

In [ ]:
for name, outcome in results.items():
    print(f"{name:32} {outcome}")